In [1]:
print("Hii")

Hii


In [8]:

# Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.feature_extraction.text import TfidfVectorizer
import xgboost as xgb
import pickle

# Display settings
pd.set_option('display.max_columns', None)
sns.set_style('darkgrid')

In [10]:

# Load Dataset

df = pd.read_csv('C:/Users/USER/Downloads/archive (1)/properties.csv')
print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nMissing Values:\n{df.isnull().sum().sort_values(ascending=False)}")

Dataset Shape: (203874, 26)

Columns: ['ad_title', 'ad_description', 'details', 'slug', 'title', 'type', 'price', 'timestamp', 'posted_date', 'deactivation_date', 'category', 'parent_category', 'location', 'geo_region', 'area', 'is_delivery_free', 'is_doorstep_delivery', 'is_dsd_applicable', 'is_member', 'is_authorized_dealer', 'is_featured_member', 'is_verified', 'membership_level', 'member_since', 'properties', 'user']

Missing Values:
member_since            26459
is_authorized_dealer     4462
is_featured_member       4462
is_member                4462
price                    1562
details                  1527
deactivation_date           6
posted_date                 6
timestamp                   6
ad_description              2
title                       0
slug                        0
type                        0
ad_title                    0
category                    0
parent_category             0
location                    0
geo_region                  0
is_dsd_applicable 

In [11]:

#Initial Data Overview

print("First 5 rows:")
display(df.head())

print("\nData Types:")
print(df.dtypes)

print("\nPrice Statistics:")
print(df['price'].describe())

print("\nProperty Types Distribution:")
print(df['type'].value_counts())

First 5 rows:


,ad_title,ad_description,details,slug,title,type,price,timestamp,posted_date,deactivation_date,category,parent_category,location,geo_region,area,is_delivery_free,is_doorstep_delivery,is_dsd_applicable,is_member,is_authorized_dealer,is_featured_member,is_verified,membership_level,member_since,properties,user
0,apartment for sale in colombo 05 | ikman,Project: Aquaria by Access Residencies.\n\nSq....,"Bedrooms: 3, Bathrooms: 2",apartment-for-sale-in-colombo-05-for-sale-colo...,apartment for sale in colombo 05,for_sale,"Rs 59,500,000",16 Feb 2:38 pm,2023-02-16T14:38:34+05:30,2022-01-31T04:43:32.000Z,Apartments For Sale,Property,Colombo 5,LK-11,"{'id': 1506, 'name': 'Colombo'}",False,False,False,True,False,False,True,plus,October 2016,"{'Address': 'kirimandala mawatha, colombo 05.'...",2a36021370cdd0f50da19f64d17ad43be687d48568ec12...
1,Land for Sale in Kurunegala | ikman.lk,à·à·à¶à·âà¶»à¶ºà·à¶±à· à¶¯à·à¶ºà·à¶«...,15.0 perches,land-for-sale-in-kurunegala-for-sale-kurunegal...,Land for Sale in Kurunegala,for_sale,"Rs 50,000 per perch",10 Nov 12:06 pm,2022-11-10T12:06:19+05:30,2022-07-30T09:30:41.000Z,Land,Property,Kurunegala City,LK-61,"{'id': 1674, 'name': 'Kurunegala'}",False,False,False,True,False,False,False,premium,April 2017,"{'Address': 'kurunegala,malkaduwawa', 'Land ty...",f917ed789e936ada5530694ede4edde85fdb53f739c5f6...
2,Land for Sale in Kelaniya | ikman,Good Residential Bare Land in Kelaniya\nLocate...,18.5 perches,land-for-sale-in-kelaniya-for-sale-gampaha-766,Land for Sale in Kelaniya,for_sale,"Rs 1,600,000 per perch",30 Jan 8:54 am,2023-01-30T08:54:35+05:30,2022-07-30T09:31:26.000Z,Land For Sale,Property,Kelaniya,LK-12,"{'id': 1577, 'name': 'Gampaha'}",False,False,False,True,False,False,True,premium,August 2016,"{'Address': 'Dalugama, Kelaniya', 'Land type':...",147c45b684bfeae3e3ffb0397b126d4e67392ad600e254...
3,Land for Sale in Kelaniya | ikman,Good Residential bare land in Kelaniya\nLocate...,12.0 perches,land-for-sale-in-kelaniya-for-sale-gampaha-765,Land for Sale in Kelaniya,for_sale,"Rs 1,700,000 per perch",30 Jan 8:54 am,2023-01-30T08:54:29+05:30,2022-07-30T09:24:20.000Z,Land For Sale,Property,Kelaniya,LK-12,"{'id': 1577, 'name': 'Gampaha'}",False,False,False,True,False,False,True,premium,August 2016,"{'Address': 'Off Kohalwila Road, Dalugama - Ke...",147c45b684bfeae3e3ffb0397b126d4e67392ad600e254...
4,Semi Luxury House for Sale in Kadawatha | ikman,Semi Luxury House in Kadawatha\nLocated at Pin...,"Bedrooms: 5, Bathrooms: 4",semi-luxury-house-for-sale-in-kadawatha-for-sa...,Semi Luxury House for Sale in Kadawatha,for_sale,"Rs 55,000,000",30 Jan 8:54 am,2023-01-30T08:54:09+05:30,2021-10-03T08:33:36.000Z,Houses For Sale,Property,Kadawatha,LK-12,"{'id': 1577, 'name': 'Gampaha'}",False,False,False,True,False,False,True,premium,August 2016,"{'Address': 'Pinthaliya Road, Kadawatha', 'Bed...",147c45b684bfeae3e3ffb0397b126d4e67392ad600e254...



Data Types:
ad_title                   str
ad_description             str
details                    str
slug                       str
title                      str
type                       str
price                      str
timestamp                  str
posted_date                str
deactivation_date          str
category                   str
parent_category            str
location                   str
geo_region                 str
area                       str
is_delivery_free          bool
is_doorstep_delivery      bool
is_dsd_applicable         bool
is_member               object
is_authorized_dealer    object
is_featured_member      object
is_verified               bool
membership_level           str
member_since               str
properties                 str
user                       str
dtype: object

Price Statistics:
count                   202312
unique                    6286
top       Rs 550,000 per perch
freq                      2830
Name: price, dtype: obje

In [13]:
# ============================================
# Cell 4: Parse 'properties' Column
# ============================================
import json

def parse_properties(prop_str):
    """Extract features from properties JSON/dict string"""
    if pd.isna(prop_str):
        return {}
    try:
        if isinstance(prop_str, str):
            # Replace single quotes with double quotes for valid JSON
            prop_dict = json.loads(prop_str.replace("'", '"'))
        else:
            prop_dict = prop_str
        return prop_dict
    except:
        return {}

# Parse properties into separate dataframe
props_df = df['properties'].apply(parse_properties).apply(pd.Series)

# Display extracted columns
print("Extracted Property Features:")
print(props_df.columns.tolist())
print(f"\nShape: {props_df.shape}")

# Display first few rows
display(props_df.head(10))

# Check missing values in parsed properties
print("\nMissing Values in Property Features:")
print(props_df.isnull().sum())

Extracted Property Features:
['Address', 'Bedrooms', 'Bathrooms', 'Size', 'Land type', 'Land size', 'House size', 'Beds', 'Baths', 'Property type', 'Features']

Shape: (203874, 11)


,Address,Bedrooms,Bathrooms,Size,Land type,Land size,House size,Beds,Baths,Property type,Features
0,"kirimandala mawatha, colombo 05.",3,2,"1,363 sqft",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"kurunegala,malkaduwawa",NaN,NaN,NaN,"Commercial, Residential",15.0 perches,NaN,NaN,NaN,NaN,NaN
2,"Dalugama, Kelaniya",NaN,NaN,NaN,"Residential, Other",18.5 perches,NaN,NaN,NaN,NaN,NaN
3,"Off Kohalwila Road, Dalugama - Kelaniya",NaN,NaN,NaN,"Residential, Other",12.0 perches,NaN,NaN,NaN,NaN,NaN
4,"Pinthaliya Road, Kadawatha",5,4,NaN,NaN,27.5 perches,"3,000.0 sqft",NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Emperor,NaN,NaN,"1,600 sqft",NaN,NaN,NaN,3,3,NaN,NaN
9,kadawatha Road,4,4,NaN,NaN,8.75 perches,"3,000.0 sqft",NaN,NaN,NaN,NaN



Missing Values in Property Features:
Address           62109
Bedrooms         150149
Bathrooms        150149
Size             184755
Land type        117777
Land size         54863
House size       140960
Beds             175849
Baths            175849
Property type    190383
Features         200330
dtype: int64
